# Hoofdstuk 7: Chattoepassingen bouwen
## Microsoft Foundry Models API Quickstart

Deze notebook is aangepast van de [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) die notebooks bevat die toegang hebben tot [Azure OpenAI](notebook-azure-openai.ipynb) services.

> **Opmerking:** GitHub Models wordt beëindigd eind juli 2026. Deze notebook is nu gericht op [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), dat hetzelfde gratis te proberen modelcatalogus en Azure AI Inference SDK-ervaring biedt.


# Overzicht  
"Grote taalmodellen zijn functies die tekst naar tekst mappen. Gegeven een invoertekststring probeert een groot taalmodel de tekst te voorspellen die daarna komt"(1). Deze "quickstart" notebook zal gebruikers introduceren in hoog-niveau LLM-concepten, kernpakketvereisten om aan de slag te gaan met AML, een zachte introductie tot promptontwerp, en verschillende korte voorbeelden van diverse gebruiksscenario's. 


## Inhoudsopgave  

[Overzicht](#overview)  
[Hoe gebruik je OpenAI Service](#how-to-use-openai-service)  
[1. Je OpenAI Service aanmaken](#1.-creating-your-openai-service)  
[2. Installatie](#2.-installation)    
[3. Referenties](#3.-credentials)  

[Toepassingen](#use-cases)    
[1. Tekst samenvatten](#1.-summarize-text)  
[2. Tekst classificeren](#2.-classify-text)  
[3. Nieuwe productnamen genereren](#3.-generate-new-product-names)  
[4. Een classifier fijn afstemmen](#4.fine-tune-a-classifier)  

[Referenties](#references)


### Maak je eerste prompt  
Deze korte oefening biedt een basisintroductie voor het indienen van prompts bij een model in Microsoft Foundry Models voor een eenvoudige taak "samenvatten".


**Stappen**:  
1. Installeer de `azure-ai-inference` bibliotheek in je Python-omgeving, als je dat nog niet hebt gedaan.  
2. Laad de standaard hulpprogramma's en stel je Microsoft Foundry Models-referenties in.  
3. Kies een model voor je taak  
4. Maak een eenvoudige prompt voor het model  
5. Dien je verzoek in bij de model-API!


### 1. Installeer `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Importeer hulpbibliotheken en maak referenties aan


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Het juiste model vinden  
Modellen zoals GPT-4o en GPT-4o mini kunnen natuurlijke taal begrijpen en genereren, en zijn beschikbaar in de Microsoft Foundry Models-catalogus naast modellen van Meta, Mistral, Cohere en Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Promptontwerp  

"De magie van grote taalmodellen is dat door getraind te worden om deze voorspellingsfout te minimaliseren over enorme hoeveelheden tekst, de modellen uiteindelijk concepten leren die nuttig zijn voor deze voorspellingen. Ze leren bijvoorbeeld concepten zoals"(1):

* hoe te spellen
* hoe grammatica werkt
* hoe te parafraseren
* hoe vragen te beantwoorden
* hoe een gesprek te voeren
* hoe in veel talen te schrijven
* hoe te programmeren
* enzovoort.

#### Hoe een groot taalmodel te besturen  
"Van alle invoer voor een groot taalmodel is verreweg de meest invloedrijke de tekstprompt(1).

Grote taalmodellen kunnen op een paar manieren worden geprikkeld om output te produceren:

Instructie: Vertel het model wat je wilt
Voltooiing: Zet het model ertoe aan om het begin van wat je wilt te voltooien
Demonstratie: Laat het model zien wat je wilt, met ofwel:
Een paar voorbeelden in de prompt
Honderden of duizenden voorbeelden in een fijn afgesteld trainingsdataset"



#### Er zijn drie basisrichtlijnen voor het maken van prompts:

**Toon en vertel**. Maak duidelijk wat je wilt, hetzij via instructies, voorbeelden, of een combinatie van beide. Als je wilt dat het model een lijst met items alfabetisch rangschikt of een alinea classificeert op sentiment, laat dan zien dat dat is wat je wilt.

**Voorzie kwalitatieve data**. Als je probeert een classifier te bouwen of het model een patroon wilt laten volgen, zorg dan dat er genoeg voorbeelden zijn. Zorg ervoor dat je je voorbeelden nakijkt — het model is meestal slim genoeg om basale spelfouten te herkennen en een reactie te geven, maar het kan ook aannemen dat dit opzettelijk is en dit kan de reactie beïnvloeden.

**Controleer je instellingen.** De instellingen temperature en top_p bepalen hoe deterministisch het model is in het genereren van een antwoord. Als je het vraagt om een antwoord waarbij er maar één juist antwoord is, dan wil je deze lager instellen. Als je meer diverse antwoorden wilt, wil je ze waarschijnlijk hoger instellen. De grootste fout die mensen maken met deze instellingen is aannemen dat ze "slimheid" of "creativiteit" regelen.


Bron: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Indienen!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Herhaal dezelfde oproep, hoe vergelijken de resultaten? 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Tekst Samenvatten  
#### Uitdaging  
Vat tekst samen door een 'tl;dr:' toe te voegen aan het einde van een tekstpassage. Let op hoe het model begrijpt hoe het een aantal taken moet uitvoeren zonder aanvullende instructies. Je kunt experimenteren met meer beschrijvende prompts dan tl;dr om het gedrag van het model te wijzigen en de samenvatting die je ontvangt aan te passen(3).  

Recente werkzaamheden hebben aanzienlijke vooruitgang laten zien bij veel NLP-taken en benchmarks door vooraf te trainen op een grote tekstcorpus, gevolgd door het fijn afstemmen op een specifieke taak. Hoewel dit doorgaans architectuur-onafhankelijk is, vereist deze methode nog steeds taak-specifieke fijn afstemming datasets van duizenden of tienduizenden voorbeelden. Daarentegen kunnen mensen doorgaans een nieuwe taaltaak uitvoeren met slechts enkele voorbeelden of eenvoudige instructies - iets waar huidige NLP-systemen nog grotendeels moeite mee hebben. Hier tonen we aan dat het opschalen van taalmodellen de taak-onafhankelijke few-shot prestaties sterk verbetert, soms zelfs concurrerend met vorige state-of-the-art fijn afstemmingsbenaderingen.  



Tl;dr  


# Oefeningen voor verschillende toepassingen  
1. Tekst samenvatten  
2. Tekst classificeren  
3. Nieuwe productnamen genereren


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Tekst classificeren  
#### Uitdaging  
Classificeer items in categorieën die tijdens de inferentie worden gegeven. In het volgende voorbeeld geven we zowel de categorieën als de te classificeren tekst in de prompt (*playground_reference). 

Klantvraag: Hallo, een van de toetsen op mijn laptoptoetsenbord is onlangs kapot gegaan en ik heb een vervanging nodig:

Geclassificeerde categorie:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Genereer Nieuwe Productnamen
#### Uitdaging
Maak productnamen op basis van voorbeeldwoorden. Hier voegen we in de prompt informatie toe over het product waarvoor we namen gaan genereren. We geven ook een soortgelijk voorbeeld om het patroon te laten zien dat we willen ontvangen. We hebben de temperatuurwaarde ook hoog ingesteld om meer willekeur en innovatie in de antwoorden te krijgen.

Productbeschrijving: Een thuis milkshake maker
Zaadwoorden: snel, gezond, compact.
Productnamen: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Productbeschrijving: Een paar schoenen die op elke voetmaat passen.
Zaadwoorden: aanpasbaar, passend, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Referenties  
- [Openai Kookboek](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Beste praktijken voor het fijn afstemmen van GPT-modellen om tekst te classificeren](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Voor Meer Hulp  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Bijdragers
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
